# 07 · Submission scripts and the setup record

The last two steps of a setup (`posydon/grids/setup/submit.py`, `record.py`):

- **the command line**: everything a run needs is passed to `posydon-run-grid` on one command line (`command_line`). `posydon-run-grid` itself is unchanged in this PR; `posydon-grid run` forwards to it;
- **the scripts** around it, by `[submission] backend`:

| backend | scripts | how the points run |
|---|---|---|
| `shell` (`job_array = true`, default) | `grid_command.sh` | a loop, one point per call |
| `shell` (`job_array = false`) | `grid_command.sh` | one `mpirun` over all points |
| `slurm_array` | `job_array_grid_submit.slurm`, `cleanup.slurm`, `run_grid.sh` | one array task per point |
| `slurm_mpi` | `mpi_grid_submit.slurm`, `cleanup.slurm`, `run_grid.sh` | one MPI job |

- **the record**: `setup_record.ini` says what the run directory was set up from.

We use `steps.prepare` to get a checked config, then call the pieces ourselves.

In [ ]:
from nb_env import env, fresh, write, show, short, grid_csv
ENV = env()

import os
from pathlib import Path
from posydon.grids.setup.steps import prepare, check_config
from posydon.grids.setup.build import build_inlists
from posydon.grids.setup.extras import stage_column_lists
from posydon.grids.setup.submit import (
    RUN_GRID_OPTIONS, command_line, find_run_grid, one_point_per_task, write_scripts)
from posydon.grids.setup.record import record_text

folder = fresh("07_submit")
grid_csv(folder)
ini = write(folder / "grid.ini", f"""
    [grid]
    type = CO-HMS_RLO
    points = grid.csv
    run_directory = run
    zams_star1 = {ENV.inlists}/r11701/ZAMS_models/zams_z1.42m2_y0.2703.data

    [scenario]
    release = v2.0
    inlist_repo = {ENV.inlists}

    [submission]
    backend = slurm_array
    account = my_account
    partition = public-cpu
    walltime = 2-00:00:00
    email = me@example.org
    cpus_per_task = 4
    newgroup = posydon_group
    """)
ready = prepare(ini)
config = ready.config
run = Path(ready.run_directory)
inlists = build_inlists(config, run, ready.parameters, scenario=ready.scenario)
columns = stage_column_lists(ready.columns, run)
executables = (str(run / "binary" / "binary"), str(run / "star1" / "star"), str(run / "star2" / "star"))

def tidy(text):
    """Shorter paths, for reading."""
    return text.replace(str(folder) + "/", "").replace(str(ENV.root), "<POSYDON>").replace(str(ENV.mesa), "<MESA_DIR>")

## The `posydon-run-grid` command line

One option per line, to read it:

In [2]:
command = command_line(config, inlists, executables, columns, run, find_run_grid())
print(tidy(command).replace(" --", " \\\n    --"))

python <POSYDON>/bin/posydon-run-grid \
    --mesa-grid grid.csv \
    --mesa-binary-executable run/binary/binary \
    --mesa-star1-executable run/star1/star \
    --mesa-star2-executable run/star2/star \
    --mesa-binary-inlist-project run/binary/inlist_project \
    --mesa-binary-inlist1 run/binary/inlist1 \
    --mesa-binary-inlist2 run/binary/inlist2 \
    --mesa-star1-inlist-project None \
    --mesa-star2-inlist-project None \
    --mesa-star-history-columns run/column_lists/history_columns.list \
    --mesa-binary-history-columns run/column_lists/binary_history_columns.list \
    --mesa-profile-columns run/column_lists/profile_columns.list \
    --output-directory run \
    --grid-type fixed \
    --psycris-inifile None \
    --grid-point-index $SLURM_ARRAY_TASK_ID \
    --job_end $SLURM_JOB_END_TIME


A few things to notice:

- `--grid-point-index $SLURM_ARRAY_TASK_ID`: one point per task;
- `--job_end $SLURM_JOB_END_TIME`: SLURM backends only;
- `--mesa-star2-inlist-project None`, `--psycris-inifile None`: `posydon-run-grid` expects every option, so unused ones are written as the string `None`. Changing that is part of the `posydon-run-grid` rewrite (tracker C11, step 7).

## The scripts, per backend

In [3]:
def scripts_for(backend, job_array=True):
    config.submission.backend = backend
    config.submission.job_array = job_array
    command = command_line(config, inlists, executables, columns, run, find_run_grid())
    out = fresh(f"07_submit_scripts/{backend}{'' if job_array else '_mpi'}")
    return write_scripts(config, command, len(ready.grid_df), out, "/opt/mesasdk", "/opt/mesa")

slurm_array = scripts_for("slurm_array")
[p.name for p in slurm_array]

['job_array_grid_submit.slurm', 'cleanup.slurm', 'run_grid.sh']

In [4]:
header, _, call = tidy(slurm_array[0].read_text()).rpartition("\n")
print(header)
print(call.replace(" --", " \\\n    --"))

#!/bin/bash
#SBATCH --account=my_account
#SBATCH --partition=public-cpu
#SBATCH -N 1
#SBATCH --array=0-2
#SBATCH --cpus-per-task 4
#SBATCH --ntasks-per-node 1
#SBATCH --time=2-00:00:00
#SBATCH --job-name="mesa_grid"
#SBATCH --output=mesa_grid.%A_%a.out
#SBATCH --mail-type=ALL
#SBATCH --mail-user=me@example.org
#SBATCH --mem-per-cpu=4G

export OMP_NUM_THREADS=4

export MESASDK_ROOT=/opt/mesasdk
source $MESASDK_ROOT/bin/mesasdk_init.sh
export MESA_DIR=/opt/mesa


python <POSYDON>/bin/posydon-run-grid \
    --mesa-grid grid.csv \
    --mesa-binary-executable run/binary/binary \
    --mesa-star1-executable run/star1/star \
    --mesa-star2-executable run/star2/star \
    --mesa-binary-inlist-project run/binary/inlist_project \
    --mesa-binary-inlist1 run/binary/inlist1 \
    --mesa-binary-inlist2 run/binary/inlist2 \
    --mesa-star1-inlist-project None \
    --mesa-star2-inlist-project None \
    --mesa-star-history-columns run/column_lists/history_columns.list \
    --mesa-binary-histo

In [5]:
show(slurm_array[1])   # cleanup.slurm: compress, hand to newgroup

#!/bin/bash
#SBATCH --account=my_account
#SBATCH --partition=public-cpu
#SBATCH -N 1
#SBATCH --cpus-per-task 1
#SBATCH --ntasks-per-node 1
#SBATCH --time=2-00:00:00
#SBATCH --job-name="mesa_grid_cleanup"
#SBATCH --output=mesa_cleanup.out
#SBATCH --mail-type=ALL
#SBATCH --mail-user=me@example.org
#SBATCH --mem-per-cpu=4G

compress-mesa .
echo "Change group to posydon_group"
chgrp -fR posydon_group .
echo "Change group permission to rwX at least"
chmod -fR g+rwX .

echo "Done."


In [6]:
show(slurm_array[2])   # run_grid.sh: submit both, cleanup after the grid

#!/bin/bash
ID_GRID=$(sbatch --parsable job_array_grid_submit.slurm)
echo "job_array_grid_submit.slurm submitted as "${ID_GRID}
ID_cleanup=$(sbatch --parsable --dependency=afterany:${ID_GRID} --kill-on-invalid-dep=yes cleanup.slurm)
echo "cleanup.slurm submitted as "${ID_cleanup}


`slurm_mpi` runs all points in one MPI job (what a dynamic grid needs):

In [7]:
slurm_mpi = scripts_for("slurm_mpi")
print([p.name for p in slurm_mpi])
print(tidy(slurm_mpi[0].read_text()).split("\n\n")[0])

['mpi_grid_submit.slurm', 'cleanup.slurm', 'run_grid.sh']
#!/bin/bash
#SBATCH --account=my_account
#SBATCH --partition=public-cpu
#SBATCH -N 1
#SBATCH --cpus-per-task 4
#SBATCH --ntasks-per-node 1
#SBATCH --time=2-00:00:00
#SBATCH --output="mesa_grid.out"
#SBATCH --mail-type=ALL
#SBATCH --mail-user=me@example.org
#SBATCH --mem-per-cpu=4G


`shell` writes one script. With `job_array = true` it loops over the points; the command is on its own line, and `compress-mesa` comes after it (fixed S13: before, it was glued to the last argument).

In [8]:
shell = scripts_for("shell")
text = tidy(shell[0].read_text())
print(text[:text.index("python")] + "python <POSYDON>/bin/posydon-run-grid ... --grid-point-index $SLURM_ARRAY_TASK_ID ; done")
print(text[text.index("compress-mesa"):])

#!/bin/bash

export OMP_NUM_THREADS=4

export MESASDK_ROOT=/opt/mesasdk
source $MESASDK_ROOT/bin/mesasdk_init.sh
export MESA_DIR=/opt/mesa

for SLURM_ARRAY_TASK_ID in 0 1 2 ; do python <POSYDON>/bin/posydon-run-grid ... --grid-point-index $SLURM_ARRAY_TASK_ID ; done
compress-mesa .
echo "Change group to posydon_group"
chgrp -fR posydon_group .
echo "Change group permission to rwX at least"
chmod -fR g+rwX .

echo "Done."


## Settings that cannot run are refused before anything is written

`check_config` stops on combinations that would set up a grid that never runs. The error names the setting as the ini says it: `[submission] backend` here, `[slurm] job_array` for a legacy ini.

- **S4 (pinned, needs step 7):** a fixed grid needs one point per task; `posydon-run-grid` cannot run a fixed grid as one MPI job yet;
- **S5 (fixed):** a dynamic grid samples new points on rank 0 of one MPI job, so a job array cannot run it. Before, it silently ran as a fixed grid over the CSV.

In [9]:
for backend, mode in [("slurm_mpi", "fixed"), ("slurm_array", "dynamic"), ("slurm_mpi", "dynamic")]:
    config.submission.backend = backend
    config.grid.mode = mode
    config.grid.psycris_inifile = ENV.extra / "psycris_dynamic.ini"
    try:
        check_config(config)
        print(f"{mode:8} + {backend:12} -> ok")
    except ValueError as error:
        print(f"{mode:8} + {backend:12} -> {error}")
config.grid.mode, config.submission.backend = "fixed", "slurm_array"

fixed    + slurm_mpi    -> [submission] backend: a fixed grid runs one point per task: use slurm_array, or shell with job_array = true
dynamic  + slurm_array  -> [submission] backend: a dynamic grid runs as one MPI job: use slurm_mpi, or shell with job_array = false
dynamic  + slurm_mpi    -> ok


`one_point_per_task` is the rule behind it:

In [10]:
from posydon.grids.setup.config import SubmissionSection
{(b, j): one_point_per_task(SubmissionSection(backend=b, job_array=j))
 for b in ["shell", "slurm_array", "slurm_mpi"] for j in [True, False]}

{('shell', True): True,
 ('shell', False): False,
 ('slurm_array', True): True,
 ('slurm_array', False): True,
 ('slurm_mpi', True): False,
 ('slurm_mpi', False): False}

## The setup record

`setup_record.ini` is written last. It names the ini, the POSYDON version, `$MESA_DIR`, the grid type and, for a scenario, the inlist repository and the **exact commit** it was at, so a grid can be traced to its physics even after the ini or a branch changes (tracker C13).

In [ ]:
from posydon import __version__
print(record_text(config, ready.resolved, ini, os.environ["MESA_DIR"], __version__))

## Try it

- Set `newgroup` away and compare `cleanup.slurm`.
- Set `[submission] work_dir = /scratch/$USER` and find `--temporary-directory` in the command.
- In a git clone of the inlist repository, edit a file the release uses (or check out another commit) and compare `inlist_commit`, `inlist_dirty` and `inlist_fingerprint` in the record.